# Обучаемый ранжировщик поверх Temporal-v3

Экспериментальный пилот, не замена принятой модели. GCN и оценка количества источников заморожены. Никакого чтения test/holdout. Выполняйте по одной ячейке сверху вниз; после smoke остановитесь до согласования затрат.

## 1. Setup — только код и окружение

Используется Python текущего kernel. Уже установленный PyTorch/CUDA не переустанавливается. На Colab старую чистую копию можно обновить; при локальных правках setup остановится.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/diffusion-sources')
    if not PROJECT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/1habibi/diffusion-sources-localization-', str(PROJECT)], check=True)
    if not (PROJECT / 'src/diffusion_sources/temporal_learned_cli.py').exists():
        dirty = subprocess.run(['git', '-C', str(PROJECT), 'status', '--porcelain'], check=True, text=True, capture_output=True).stdout.strip()
        if dirty:
            raise RuntimeError('STOP: checkout содержит локальные изменения. Обнови код отдельно, не теряя их.')
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
    packages = {'numpy': 'numpy', 'networkx': 'networkx', 'torch': 'torch', 'torch_geometric': 'torch-geometric', 'sklearn': 'scikit-learn', 'scipy': 'scipy', 'yaml': 'PyYAML', 'tqdm': 'tqdm'}
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT), '--no-deps'], check=True)
else:
    PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/diffusion_sources/temporal_learned_cli.py').exists()), None)
    if PROJECT is None:
        raise RuntimeError('Открой notebook из корня ветки temporal-learned-reranker.')

sys.path.insert(0, str(PROJECT / 'src'))
import torch
import diffusion_sources.temporal_learned_cli as learned
from diffusion_sources.temporal_learned_cli import LearnedPaths, check_inputs, run_stage

if not Path(learned.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise RuntimeError('Kernel импортировал другую копию проекта. Перезапусти kernel.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Kernel Python:', sys.executable)
print('Project import:', learned.__file__)
print('PyTorch:', torch.__version__, 'device:', DEVICE)
print('Setup не выполнял inference, cache или обучение.')

## 2. Пути и аутентификация

Проверяем только reference train, граф и run 7026. Validation и остальные checkpoint пока не читаются. Локально backup только читается; Colab использует исходные папки Drive.

In [ ]:
import json
if IN_COLAB:
    ROOT = Path('/content/drive/MyDrive/diffusion-sources')
    DATA = ROOT / 'data/facebook_main'
    RUN_ROOT = ROOT / 'reports/runs/facebook_main_v2'
    RUNS = {7026: RUN_ROOT / 's1b/seed_7026', 7027: RUN_ROOT / 'frozen_candidate/seed_7027', 7028: RUN_ROOT / 'frozen_candidate/seed_7028'}
else:
    MAIN_PROJECT = PROJECT.parent.parent if PROJECT.parent.name == '.worktrees' else PROJECT
    BACKUP = MAIN_PROJECT / 'reports/backups/temporal_v3_20260929'
    mapping = json.loads((BACKUP / 'local_paths.json').read_text(encoding='utf-8'))
    DATA = BACKUP / mapping['reference_data']
    RUNS = {int(seed): BACKUP / value for seed, value in mapping['runs'].items()}
    ROOT = PROJECT
PATHS = LearnedPaths(DATA, RUNS, ROOT / 'reports/runs/temporal_v3_learned_reranker/v1', PROJECT / 'configs/temporal_learned_reranker_v1.json')
CHECKED = check_inputs(PATHS, torch.device(DEVICE))
print('Data:', DATA)
print('Output:', PATHS.output_dir)
print('Train inputs verified; old targets unopened.')

## 3. Smoke — 6 train-примеров, без обучения

Проверяем replay, beta=0 и неизменность количества. Прогноз времени cache приблизительный, не гарантия. STOP после этой ячейки: перед cache/fit согласуй запуск.

In [ ]:
import json
SMOKE = run_stage('smoke', PATHS, DEVICE)
print(json.dumps(SMOKE, ensure_ascii=False, indent=2))
print('STOP: smoke не является оценкой качества. До согласования не запускай ячейку 4.')

## 4. Train cache — 2160 каскадов, только seed 7026

Fit 1620 + selection-dev 540, разделение по целым каскадам. Замороженная GCN не обучается. Готовый кэш повторно используется только при совпадении хэшей и окружения.

In [ ]:
import json
CACHE = run_stage('cache', PATHS, DEVICE)
print(json.dumps(CACHE, ensure_ascii=False, indent=2))

## 5. Три лёгких fit и выбор C на dev

Введи RUN_LEARNED_RERANKER_PILOT только после согласования эксперимента. StandardScaler обучается только на fit. Ровно C=0.1/1/10; без refit. Gate сравнивает с Temporal-v3 beta=0.5.

In [ ]:
import json
answer = input('Для согласованного fit введи RUN_LEARNED_RERANKER_PILOT: ').strip()
if answer != 'RUN_LEARNED_RERANKER_PILOT':
    raise ValueError('STOP: требуется RUN_LEARNED_RERANKER_PILOT; обучение не запущено.')
SELECTION = run_stage('select', PATHS, DEVICE)
print(json.dumps({key: SELECTION.get(key) for key in ('selected_C', 'gate', 'head', 'fit_rows', 'elapsed_seconds')}, ensure_ascii=False, indent=2))
print('Полные dev-метрики и три C сохранены в selection/payload.json.')

## 6. Primary validation — только после dev gate

Все 1998 validation-примеров. Проверка saved snapshot F1, парный CI и групповые условия. Это exploratory оценка: validation уже участвовала в разработке.

In [ ]:
import json
if not globals().get('SELECTION', {}).get('gate', {}).get('passed', False):
    VALIDATION = {'status': 'blocked'}
    print('STOP: сначала ячейка 5 и passed dev gate. Validation не читается.')
else:
    VALIDATION = run_stage('validate', PATHS, DEVICE)
    print(json.dumps({key: VALIDATION.get(key) for key in ('seed', 'delta_f1', 'delta_exact', 'f1_ci', 'delta_by_k', 'delta_by_candidates', 'gate', 'status')}, ensure_ascii=False, indent=2))

## 7. Frozen repeats 7027/7028 — только после primary pass

Та же единственная линейная модель, тот же набор наблюдений. Никакого fit или изменения C между seed.

In [ ]:
import json
if not globals().get('VALIDATION', {}).get('gate', {}).get('passed', False):
    CONFIRM = {'status': 'blocked'}
    print('STOP: primary gate не пройден. Повторы не запускаются.')
else:
    CONFIRM = run_stage('confirm', PATHS, DEVICE)
    print(json.dumps({key: CONFIRM.get(key) for key in ('status', 'seeds', 'delta_f1_by_seed', 'elapsed_seconds')}, ensure_ascii=False, indent=2))

## 8. Summary — решение о перспективности, не deployment

Bootstrap по 1998 усреднённым каскадам, не по 5994 независимым строкам. Passed означает только повод обсудить новую независимую проверку. Старый holdout не открывается.

In [ ]:
import json
if globals().get('CONFIRM', {}).get('status') != 'complete':
    SUMMARY = {'status': 'blocked'}
    print('STOP: нет завершённых primary и двух repeats.')
else:
    SUMMARY = run_stage('summary', PATHS, DEVICE)
    print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))